# Mumbai Data Preprocessing

Combine EXIF metadata (video info + GPS timeseries) with annotation data to create a merged dataset.

In [1]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

In [2]:
BASE_DIR = Path('../..')

VIDEO_METADATA_FILES = [
    BASE_DIR / '1_8_exif_video_metadata.csv',
    BASE_DIR / '9_11_exif_video_metadata.csv',
    BASE_DIR / '13_19_6_7_exif_video_metadata.csv',
]

GPS_TIMESERIES_FILES = [
    BASE_DIR / '1_8_gps_timeseries.csv',
    BASE_DIR / '9_11_gps_timeseries.csv',
    BASE_DIR / '13_19_6_7_gps_timeseries_new.csv',
]

ANNOTATION_FILE = BASE_DIR / 'labelstudio/mumbai_export_221408_project-221408-at-2026-02-02-21-33-2247d6c5.json'

FRAME_METADATA_FILES = [
    BASE_DIR / '1_8_frame_metadata.log',
    BASE_DIR / '9_11_frame_metadata.log',
    BASE_DIR / '13_19_6_7_frame_metadata.log',
]

OUTPUT_FILE = BASE_DIR / 'data/mumbai_annotations_with_exif.csv'

## 1. Load Video Metadata

In [3]:
video_dfs = []
for f in VIDEO_METADATA_FILES:
    if f.exists():
        df = pd.read_csv(f)
        video_dfs.append(df)
        print(f"{f.name}: {len(df)} videos")

video_metadata = pd.concat(video_dfs, ignore_index=True)
print(f"\nTotal videos: {len(video_metadata)}")
print(f"Columns: {list(video_metadata.columns)}")

1_8_exif_video_metadata.csv: 85 videos
9_11_exif_video_metadata.csv: 40 videos
13_19_6_7_exif_video_metadata.csv: 49 videos

Total videos: 174
Columns: ['video_id', 'source_folder', 'video_name', 'original_video_filename', 'unique_video_filename', 'file_size_bytes', 'file_size_mb', 'file_created_at', 'file_modified_at', 'video_duration_sec', 'video_fps', 'video_resolution_wxh', 'video_codec', 'camera_model', 'recording_datetime', 'exif_filename', 'exif_file_path', 'exif_relative_path']


In [4]:
video_metadata[['video_id', 'recording_datetime', 'video_duration_sec', 'camera_model']].head(10)

,video_id,recording_datetime,video_duration_sec,camera_model
0,1_itinerary_2_2_c20922c0,2025:03:27 04:02:07,NaN,HERO13 Black
1,2_itinerary_1_2_d78a4abf,2025:03:29 02:46:21,107.0,HERO13 Black
2,1_itinerary_8_1_9f93eb04,2025:03:27 09:46:22,142.0,HERO13 Black
3,1_itinerary_2_3_4ea846af,2025:03:27 04:04:51,157.0,HERO13 Black
4,2_itinerary_2_2_184f0b4c,2025:03:29 03:36:56,53.0,HERO13 Black
5,2_itinerary_5_2_eeef3957,2025:03:29 09:01:36,289.0,HERO13 Black
6,1_itinerary_1_2_f6dfb69b,2025:03:27 03:09:56,479.0,HERO13 Black
7,2_dadar_flower_market_bbbf412d,2025:03:29 11:00:44,587.0,HERO13 Black
8,1_itinerary_1_1_8e5f0fc4,2025:03:27 02:56:43,602.0,HERO13 Black
9,1_itinerary_8_2_44f36245,2025:03:27 09:49:43,611.0,HERO13 Black


## 2. Load GPS Timeseries

In [5]:
def dms_to_decimal(dms_str):
    """Convert DMS format to decimal degrees."""
    if pd.isna(dms_str):
        return None
    match = re.match(r"(\d+) deg (\d+)' ([\d.]+)\" ([NSEW])", str(dms_str))
    if match:
        d, m, s, direction = match.groups()
        decimal = float(d) + float(m)/60 + float(s)/3600
        if direction in ['S', 'W']:
            decimal = -decimal
        return decimal
    return None

def parse_altitude(alt_str):
    """Parse altitude string like '11.815 m' to float."""
    if pd.isna(alt_str):
        return None
    match = re.match(r"([\d.-]+)\s*m", str(alt_str))
    if match:
        return float(match.group(1))
    return None

In [6]:
gps_dfs = []
for f in GPS_TIMESERIES_FILES:
    if f.exists():
        df = pd.read_csv(f)
        gps_dfs.append(df)
        print(f"{f.name}: {len(df):,} GPS points")

gps_timeseries = pd.concat(gps_dfs, ignore_index=True)
print(f"\nTotal GPS points: {len(gps_timeseries):,}")

1_8_gps_timeseries.csv: 630,753 GPS points
9_11_gps_timeseries.csv: 333,067 GPS points
13_19_6_7_gps_timeseries_new.csv: 426,313 GPS points

Total GPS points: 1,390,133


In [7]:
gps_timeseries['lat'] = gps_timeseries['gps_latitude'].apply(dms_to_decimal)
gps_timeseries['lon'] = gps_timeseries['gps_longitude'].apply(dms_to_decimal)
gps_timeseries['alt'] = gps_timeseries['gps_altitude'].apply(parse_altitude)

gps_timeseries['gps_datetime'] = pd.to_datetime(gps_timeseries['gps_datetime'], format='%Y:%m:%d %H:%M:%S.%f', errors='coerce')

print(f"GPS points with valid lat/lon: {gps_timeseries['lat'].notna().sum():,}")
gps_timeseries[['video_id', 'gps_datetime', 'lat', 'lon', 'alt']].head()

GPS points with valid lat/lon: 1,390,133


,video_id,gps_datetime,lat,lon,alt
0,1_itinerary_1_1_8e5f0fc4,2025-03-27 02:55:23.899,19.110911,73.006186,11.815
1,1_itinerary_1_1_8e5f0fc4,2025-03-27 02:55:24.000,19.110911,73.006186,11.815
2,1_itinerary_1_1_8e5f0fc4,2025-03-27 02:55:24.099,19.110911,73.006186,11.815
3,1_itinerary_1_1_8e5f0fc4,2025-03-27 02:55:24.199,19.110911,73.006186,11.815
4,1_itinerary_1_1_8e5f0fc4,2025-03-27 02:55:24.299,19.110911,73.006186,11.815


## 3. Load Frame Metadata

In [8]:
def parse_frame_metadata_log(filepath):
    """Parse pipe-delimited frame metadata log file."""
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('|')
            if len(parts) >= 10:
                rows.append({
                    'base_video_id': parts[0],
                    'source_folder': parts[1],
                    'video_name': parts[2],
                    'original_filename': parts[3],
                    'frame_filename': parts[4],
                    'frame_number_raw': int(parts[5]),
                    'timestamp_sec': float(parts[6]),
                    'timestamp_str': parts[7],
                    'fps': float(parts[8]),
                    'video_duration_sec': float(parts[9])
                })
    return pd.DataFrame(rows)

frame_dfs = []
for f in FRAME_METADATA_FILES:
    if f.exists():
        df = parse_frame_metadata_log(f)
        frame_dfs.append(df)
        print(f"{f.name}: {len(df):,} frames")

frame_metadata = pd.concat(frame_dfs, ignore_index=True)
print(f"\nTotal frames: {len(frame_metadata):,}")

1_8_frame_metadata.log: 7,169 frames
9_11_frame_metadata.log: 3,698 frames
13_19_6_7_frame_metadata.log: 4,738 frames

Total frames: 15,605


In [9]:
frame_metadata.head()

,base_video_id,source_folder,video_name,original_filename,frame_filename,frame_number_raw,timestamp_sec,timestamp_str,fps,video_duration_sec
0,1_itinerary_1_1,1,itinerary_1_1,itinerary_1_1.MP4,1_itinerary_1_1_frame000000.jpg,0,0.0,0:00:00,29.97003,602.134867
1,1_itinerary_1_1,1,itinerary_1_1,itinerary_1_1.MP4,1_itinerary_1_1_frame000001.jpg,300,10.0,0:00:10,29.97003,602.134867
2,1_itinerary_1_1,1,itinerary_1_1,itinerary_1_1.MP4,1_itinerary_1_1_frame000002.jpg,599,20.0,0:00:20,29.97003,602.134867
3,1_itinerary_1_1,1,itinerary_1_1,itinerary_1_1.MP4,1_itinerary_1_1_frame000003.jpg,899,30.0,0:00:30,29.97003,602.134867
4,1_itinerary_1_1,1,itinerary_1_1,itinerary_1_1.MP4,1_itinerary_1_1_frame000004.jpg,1199,40.0,0:00:40,29.97003,602.134867


## 4. Load Annotations

In [10]:
with open(ANNOTATION_FILE, 'r') as f:
    annotation_data = json.load(f)

print(f"Total annotation tasks: {len(annotation_data)}")

Total annotation tasks: 2863


In [11]:
def parse_annotation(task, annotation):
    """Parse a single annotation into a flat dict."""
    row = {
        'task_id': task['id'],
        'annotation_id': annotation['id'],
        'annotator_email': annotation['completed_by']['email'],
        'image': task['data']['image'],
        'file_upload': task.get('file_upload', ''),
    }
    
    for result in annotation['result']:
        field_name = result['from_name']
        if result['type'] == 'taxonomy':
            value = result['value']['taxonomy'][0][0] if result['value']['taxonomy'] else None
        elif result['type'] == 'choices':
            value = result['value']['choices'][0] if result['value']['choices'] else None
        elif result['type'] == 'textarea':
            value = result['value']['text'][0] if result['value']['text'] else None
        else:
            value = None
        row[field_name] = value
    
    return row

rows = []
for task in annotation_data:
    for annotation in task['annotations']:
        rows.append(parse_annotation(task, annotation))

annotations = pd.DataFrame(rows)
print(f"Total annotations: {len(annotations)}")

Total annotations: 2875


In [12]:
core_fields = ['men_count', 'women_count', 'men_twowheeler', 'women_twowheeler', 
               'footpath', 'lane_markings', 'potholes', 'litter']
skip_fields = ['bus_station', 'railway_station', 'street_vendor']

def is_skip_row(row):
    has_core = any(pd.notna(row.get(f)) for f in core_fields if f in row.index)
    has_skip = all(
        f in row.index and pd.notna(row.get(f)) and row.get(f) == 'No' 
        for f in skip_fields
    )
    return (not has_core) and has_skip

skip_mask = annotations.apply(is_skip_row, axis=1)
print(f"Skip rows: {skip_mask.sum()}")
print(f"Valid annotations: {(~skip_mask).sum()}")

annotations = annotations[~skip_mask].copy()

Skip rows: 131
Valid annotations: 2744


## 5. Extract Frame Info from Image Paths

In [13]:
def extract_frame_info(image_path, frame_metadata_df=None):
    """Extract video_id, frame_number, and timestamp from image path.

    Handles two formats:
    1. upload/221408/c594e56e-3_itinerary_7_frame05100_t000250_170.jpg (with uuid, timestamp info)
    2. upload/221408/10_itinerary_11_frame000011.jpg (simple format)
    """
    filename = image_path.split('/')[-1]

    # Format 1: uuid-prefix with timestamp info
    match = re.match(r'[a-f0-9]+-(.+)_frame(\d+)_t(\d+)_(\d+)\.jpg', filename)
    if match:
        base_video_id = match.group(1)
        frame_number = int(match.group(2))
        timestamp_parts = match.group(3)
        frame_idx = int(match.group(4))

        if len(timestamp_parts) == 6:
            minutes = int(timestamp_parts[:3])
            seconds = int(timestamp_parts[3:])
            timestamp_sec = minutes * 60 + seconds
        else:
            timestamp_sec = float(timestamp_parts)

        return {
            'base_video_id': base_video_id,
            'frame_number': frame_number,
            'timestamp_sec': timestamp_sec,
            'frame_idx': frame_idx
        }

    # Format 2: Simple format without uuid prefix (e.g., 10_itinerary_11_frame000011.jpg)
    match = re.match(r'(.+)_frame(\d+)\.jpg', filename)
    if match:
        base_video_id = match.group(1)
        frame_number = int(match.group(2))
        # Calculate timestamp from frame number assuming 10 sec intervals (frame_idx * 10)
        timestamp_sec = frame_number * 10.0

        return {
            'base_video_id': base_video_id,
            'frame_number': frame_number,
            'timestamp_sec': timestamp_sec,
            'frame_idx': frame_number
        }

    return {}

frame_info = annotations['image'].apply(extract_frame_info).apply(pd.Series)
annotations = pd.concat([annotations, frame_info], axis=1)
print(f"Extracted frame info for {annotations['base_video_id'].notna().sum()} annotations")

Extracted frame info for 2744 annotations


In [14]:
annotations[['image', 'base_video_id', 'frame_number', 'timestamp_sec']].head(10)

,image,base_video_id,frame_number,timestamp_sec
0,upload/221408/c594e56e-3_itinerary_7_frame0510...,3_itinerary_7,5100,250.0
2,upload/221408/97906f71-1_itinerary_1_1_frame17...,1_itinerary_1_1,17400,940.0
3,upload/221408/da923199-1_itinerary_1_1_frame16...,1_itinerary_1_1,16800,920.0
4,upload/221408/17b36238-1_itinerary_1_1_frame16...,1_itinerary_1_1,16200,900.0
5,upload/221408/f578febd-1_itinerary_1_1_frame15...,1_itinerary_1_1,15900,850.0
6,upload/221408/48b5def0-1_itinerary_1_1_frame15...,1_itinerary_1_1,15300,830.0
7,upload/221408/48b5def0-1_itinerary_1_1_frame15...,1_itinerary_1_1,15300,830.0
8,upload/221408/12a8dfab-1_itinerary_1_1_frame15...,1_itinerary_1_1,15000,820.0
9,upload/221408/52919841-1_itinerary_1_1_frame14...,1_itinerary_1_1,14700,810.0
10,upload/221408/78817e66-1_itinerary_1_1_frame14...,1_itinerary_1_1,14400,800.0


## 6. Match Annotations to Video Metadata

In [15]:
def extract_video_base_id(video_id):
    """Extract base video ID without hash suffix.
    
    Example: 1_itinerary_1_1_8e5f0fc4 -> 1_itinerary_1_1
    """
    parts = video_id.rsplit('_', 1)
    if len(parts) == 2 and len(parts[1]) == 8:
        return parts[0]
    return video_id

video_metadata['base_video_id'] = video_metadata['video_id'].apply(extract_video_base_id)

print("Sample video_id mappings:")
print(video_metadata[['video_id', 'base_video_id']].head(10))

Sample video_id mappings:
                         video_id          base_video_id
0        1_itinerary_2_2_c20922c0        1_itinerary_2_2
1        2_itinerary_1_2_d78a4abf        2_itinerary_1_2
2        1_itinerary_8_1_9f93eb04        1_itinerary_8_1
3        1_itinerary_2_3_4ea846af        1_itinerary_2_3
4        2_itinerary_2_2_184f0b4c        2_itinerary_2_2
5        2_itinerary_5_2_eeef3957        2_itinerary_5_2
6        1_itinerary_1_2_f6dfb69b        1_itinerary_1_2
7  2_dadar_flower_market_bbbf412d  2_dadar_flower_market
8        1_itinerary_1_1_8e5f0fc4        1_itinerary_1_1
9        1_itinerary_8_2_44f36245        1_itinerary_8_2


In [16]:
annotations_with_video = annotations.merge(
    video_metadata[['video_id', 'base_video_id', 'source_folder', 'recording_datetime', 
                    'video_duration_sec', 'camera_model', 'video_fps']],
    on='base_video_id',
    how='left'
)

print(f"Annotations matched to video: {annotations_with_video['video_id'].notna().sum()}")
print(f"Annotations without video match: {annotations_with_video['video_id'].isna().sum()}")

Annotations matched to video: 2744
Annotations without video match: 0


In [17]:
if annotations_with_video['video_id'].isna().any():
    unmatched = annotations_with_video[annotations_with_video['video_id'].isna()]['base_video_id'].unique()
    print(f"Unmatched base_video_ids ({len(unmatched)}):")
    for vid in unmatched[:20]:
        print(f"  {vid}")

## 7. Compute Frame Datetime and Interpolate GPS Coordinates

In [18]:
video_metadata['recording_datetime_parsed'] = pd.to_datetime(
    video_metadata['recording_datetime'], 
    format='%Y:%m:%d %H:%M:%S', 
    errors='coerce'
)

video_start_times = video_metadata.set_index('video_id')['recording_datetime_parsed'].to_dict()

annotations_with_video['frame_datetime'] = pd.to_datetime(
    annotations_with_video['recording_datetime'],
    format='%Y:%m:%d %H:%M:%S',
    errors='coerce'
) + pd.to_timedelta(annotations_with_video['timestamp_sec'], unit='s')

annotations_with_video['frame_hour'] = annotations_with_video['frame_datetime'].dt.hour
annotations_with_video['frame_dayofweek'] = annotations_with_video['frame_datetime'].dt.dayofweek

print(f"Computed frame_datetime for {annotations_with_video['frame_datetime'].notna().sum()} annotations")
print(f"Sample frame_datetime values:")
print(annotations_with_video[['recording_datetime', 'timestamp_sec', 'frame_datetime', 'frame_hour']].head())

Computed frame_datetime for 2744 annotations
Sample frame_datetime values:
    recording_datetime  timestamp_sec      frame_datetime  frame_hour
0  2025:03:31 11:47:22          250.0 2025-03-31 11:51:32          11
1  2025:03:27 02:56:43          940.0 2025-03-27 03:12:23           3
2  2025:03:27 02:56:43          920.0 2025-03-27 03:12:03           3
3  2025:03:27 02:56:43          900.0 2025-03-27 03:11:43           3
4  2025:03:27 02:56:43          850.0 2025-03-27 03:10:53           3


In [19]:
def get_gps_for_frame_interpolated(row, gps_df):
    """Get GPS coordinates for a frame using linear interpolation between surrounding GPS points."""
    video_id = row.get('video_id')
    frame_datetime = row.get('frame_datetime')
    
    if pd.isna(video_id) or pd.isna(frame_datetime):
        return pd.Series({'gps_lat': None, 'gps_lon': None, 'gps_alt': None, 'gps_time_diff_sec': None})
    
    video_gps = gps_df[gps_df['video_id'] == video_id].copy()
    if len(video_gps) == 0:
        return pd.Series({'gps_lat': None, 'gps_lon': None, 'gps_alt': None, 'gps_time_diff_sec': None})
    
    video_gps = video_gps.sort_values('gps_datetime')
    
    before = video_gps[video_gps['gps_datetime'] <= frame_datetime]
    after = video_gps[video_gps['gps_datetime'] > frame_datetime]
    
    if len(before) > 0 and len(after) > 0:
        before_row = before.iloc[-1]
        after_row = after.iloc[0]
        
        t0 = before_row['gps_datetime']
        t1 = after_row['gps_datetime']
        
        total_interval = (t1 - t0).total_seconds()
        if total_interval > 0:
            weight = (frame_datetime - t0).total_seconds() / total_interval
        else:
            weight = 0.5
        
        lat = before_row['lat'] + weight * (after_row['lat'] - before_row['lat'])
        lon = before_row['lon'] + weight * (after_row['lon'] - before_row['lon'])
        alt = before_row['alt'] + weight * (after_row['alt'] - before_row['alt']) if pd.notna(before_row['alt']) and pd.notna(after_row['alt']) else before_row['alt']
        
        time_diff = min(
            abs((frame_datetime - t0).total_seconds()),
            abs((frame_datetime - t1).total_seconds())
        )
    elif len(before) > 0:
        before_row = before.iloc[-1]
        lat, lon, alt = before_row['lat'], before_row['lon'], before_row['alt']
        time_diff = abs((frame_datetime - before_row['gps_datetime']).total_seconds())
    elif len(after) > 0:
        after_row = after.iloc[0]
        lat, lon, alt = after_row['lat'], after_row['lon'], after_row['alt']
        time_diff = abs((frame_datetime - after_row['gps_datetime']).total_seconds())
    else:
        return pd.Series({'gps_lat': None, 'gps_lon': None, 'gps_alt': None, 'gps_time_diff_sec': None})
    
    if time_diff > 30:
        return pd.Series({'gps_lat': None, 'gps_lon': None, 'gps_alt': None, 'gps_time_diff_sec': None})
    
    return pd.Series({
        'gps_lat': lat,
        'gps_lon': lon,
        'gps_alt': alt,
        'gps_time_diff_sec': round(time_diff, 3)
    })

In [20]:
print("Matching GPS coordinates to frames using interpolation (this may take a moment)...")

gps_coords = annotations_with_video.apply(
    lambda row: get_gps_for_frame_interpolated(row, gps_timeseries),
    axis=1
)

annotations_final = pd.concat([annotations_with_video, gps_coords], axis=1)

print(f"Annotations with GPS: {annotations_final['gps_lat'].notna().sum()}")
print(f"Annotations without GPS: {annotations_final['gps_lat'].isna().sum()}")
print(f"\nGPS time diff statistics (seconds):")
print(annotations_final['gps_time_diff_sec'].describe())

Matching GPS coordinates to frames using interpolation (this may take a moment)...
Annotations with GPS: 1985
Annotations without GPS: 759

GPS time diff statistics (seconds):
count    1985.000000
mean        0.497050
std         2.937313
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        29.201000
Name: gps_time_diff_sec, dtype: float64


## 8. Convert Count Fields to Numeric

In [21]:
count_cols = ['men_count', 'women_count', 'men_twowheeler', 'women_twowheeler']

def convert_count(val):
    if pd.isna(val):
        return np.nan
    if val == '>10':
        return 11
    try:
        return int(val)
    except (ValueError, TypeError):
        return np.nan

for col in count_cols:
    if col in annotations_final.columns:
        annotations_final[col] = annotations_final[col].apply(convert_count)

## 9. Save Merged Dataset

In [22]:
output_columns = [
    'task_id', 'annotation_id', 'annotator_email', 'image',
    'base_video_id', 'video_id', 'frame_number', 'timestamp_sec',
    'source_folder', 'recording_datetime', 'frame_datetime',
    'frame_hour', 'frame_dayofweek',
    'video_duration_sec', 'camera_model',
    'gps_lat', 'gps_lon', 'gps_alt', 'gps_time_diff_sec',
    'men_count', 'women_count', 'men_twowheeler', 'women_twowheeler',
    'footpath', 'lane_markings', 'potholes', 'litter',
    'bus_station', 'railway_station', 'street_vendor'
]

available_cols = [c for c in output_columns if c in annotations_final.columns]
output_df = annotations_final[available_cols].copy()

print(f"Output columns: {len(available_cols)}")
print(f"Output rows: {len(output_df)}")
print(f"\nNew columns added: frame_datetime, frame_hour, frame_dayofweek, gps_time_diff_sec")

Output columns: 30
Output rows: 2744

New columns added: frame_datetime, frame_hour, frame_dayofweek, gps_time_diff_sec


In [23]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
output_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

Saved to ../../data/mumbai_annotations_with_exif.csv


## 10. Summary

In [24]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"\nInput:")
print(f"  Videos in metadata: {len(video_metadata)}")
print(f"  GPS points: {len(gps_timeseries):,}")
print(f"  Annotations: {len(annotations)}")

print(f"\nOutput:")
print(f"  Total rows: {len(output_df)}")
print(f"  With video metadata: {output_df['video_id'].notna().sum()}")
print(f"  With GPS coordinates: {output_df['gps_lat'].notna().sum()}")
print(f"  With frame_datetime: {output_df['frame_datetime'].notna().sum()}")

print(f"\nGPS Matching Quality:")
if output_df['gps_time_diff_sec'].notna().any():
    print(f"  Mean time diff: {output_df['gps_time_diff_sec'].mean():.3f} sec")
    print(f"  Max time diff: {output_df['gps_time_diff_sec'].max():.3f} sec")
    print(f"  Points with <1 sec diff: {(output_df['gps_time_diff_sec'] < 1).sum()}")

print(f"\nGPS Coverage:")
if output_df['gps_lat'].notna().any():
    print(f"  Lat range: {output_df['gps_lat'].min():.4f} to {output_df['gps_lat'].max():.4f}")
    print(f"  Lon range: {output_df['gps_lon'].min():.4f} to {output_df['gps_lon'].max():.4f}")

print(f"\nTemporal Coverage:")
if output_df['frame_hour'].notna().any():
    print(f"  Hour range: {output_df['frame_hour'].min()} to {output_df['frame_hour'].max()}")

PREPROCESSING SUMMARY

Input:
  Videos in metadata: 174
  GPS points: 1,390,133
  Annotations: 2744

Output:
  Total rows: 2744
  With video metadata: 2744
  With GPS coordinates: 1985
  With frame_datetime: 2744

GPS Matching Quality:
  Mean time diff: 0.497 sec
  Max time diff: 29.201 sec
  Points with <1 sec diff: 1913

GPS Coverage:
  Lat range: 18.9104 to 19.1110
  Lon range: 72.7954 to 73.0121

Temporal Coverage:
  Hour range: 1 to 14


In [25]:
output_df[['frame_datetime', 'frame_hour', 'gps_lat', 'gps_lon', 'gps_time_diff_sec', 'men_count', 'women_count']].head(10)

,frame_datetime,frame_hour,gps_lat,gps_lon,gps_time_diff_sec,men_count,women_count
0,2025-03-31 11:51:32,11,19.006764,72.835172,0.0,3.0,2.0
1,2025-03-27 03:12:23,3,NaN,NaN,NaN,NaN,NaN
2,2025-03-27 03:12:03,3,NaN,NaN,NaN,8.0,7.0
3,2025-03-27 03:11:43,3,NaN,NaN,NaN,11.0,1.0
4,2025-03-27 03:10:53,3,NaN,NaN,NaN,4.0,2.0
5,2025-03-27 03:10:33,3,NaN,NaN,NaN,1.0,1.0
6,2025-03-27 03:10:33,3,NaN,NaN,NaN,2.0,NaN
7,2025-03-27 03:10:23,3,NaN,NaN,NaN,NaN,NaN
8,2025-03-27 03:10:13,3,NaN,NaN,NaN,NaN,NaN
9,2025-03-27 03:10:03,3,NaN,NaN,NaN,3.0,4.0
